## 课程四：MeloTTS 的训练与推理优化（加分项：10%，任选一道即可）

说明：从以下两道题目任选一道完成，提交时请注明选择的题目。选题一包含训练过程，若资源有限可选择选题二。

## 选题一：MeloTTS 的说话人音色训练

实验目标：通过使用 MeloTTS 模型进行说话人音色训练，掌握模型的训练流程和参数调整方法，并学习如何评估 TTS 模型的性能。

请参考 MeloTTS 的官方 [训练教程](https://github.com/myshell-ai/MeloTTS/blob/main/docs/training.md)，从 [原神语音数据集](https://www.bilibili.com/read/cv36652528) 中任选一个角色语音作为数据集，训练一个新的中文说话人音色 MeloTTS 模型。并从数据集中至少划分 30 条语音作为测试集，使用 Whisper 或其他 ASR 模型评测 CER，WavLM 模型评测说话人相似度。

参考资料：
1. MeloTTS 官方训练教程：https://github.com/myshell-ai/MeloTTS/blob/main/docs/training.md
2. MeloTTS 中文说话人音色训练教程：https://blog.lukeewin.top/archives/melotts-zh-model-train
2. 原神语音数据集使用说明：https://www.bilibili.com/read/cv36652528
3. 数据集下载地址：https://res.acgnai.top
4. Whisper 模型仓库及使用示例：https://github.com/openai/whisper
5. 说话人相似度评测模型和示例：https://huggingface.co/microsoft/wavlm-base-plus-sv

预训练模型：  
由于官方模型是基于英文数据集训练的，直接使用可能会导致中文语音合成效果不佳或训练速度较慢。可以使用以下预训练的中文模型进行微调：
1. Hugging Face：https://huggingface.co/CJY/MeloTTS-ZH-BZNSYP
2. 国内可使用 ModelScope：https://www.modelscope.cn/models/CJY1018/MeloTTS-ZH-BZNSYP

要求：
1. 说明选择的说话人名称、训练集时长，时长 ≥ 10分钟
2. 简要说明训练过程，包含 loss 变化截图
3. 提供测试集的 CER 和 说话人相似度评测结果

## 选题二：MeloTTS 的推理优化

实验目标：学习使用 sherpa-onnx 模型推理优化工具，将 MeloTTS 模型导出为 ONNX 格式，并对比原始模型和 ONNX 优化模型在推理速度的优势和生成语音质量的差异。

请参考 k2-fsa/sherpa-onnx 的 [代码仓库](https://github.com/k2-fsa/sherpa-onnx) 和 [教程文档](https://k2-fsa.github.io/sherpa/onnx/tts/pretrained_models/vits.html)，将原始的 MeloTTS 模型进行 ONNX 模型导出优化，并与原始模型进行对比，展示推理速度的提升、wer/cer 和语音质量的对比。

参考资料：
1. k2-fsa/sherpa-onnx 仓库：https://github.com/k2-fsa/sherpa-onnx, 
2. MeloTTS 的 ONNX 模型导出代码示例：https://github.com/k2-fsa/sherpa-onnx/tree/master/scripts/melo-tts，在 `scripts/melo-tts` 目录下
3. sherpa-onnx 教程文档：https://k2-fsa.github.io/sherpa/onnx/tts/pretrained_models/vits.html
4. Whisper 模型仓库及使用示例：https://github.com/openai/whisper
5. 语音质量客观指标 DNSMOS 及评测示例：https://github.com/microsoft/DNS-Challenge 、https://pypi.org/project/speechmos

要求：
1. 调研并说明 ONNX 模型在推理优化中的核心原理、优势和适用场景
2. 简要说明使用 sherpa-onnx 对 MeloTTS 进行 ONNX 模型导出的过程和关键步骤
3. 在同一硬件环境下对比原始模型和 ONNX 优化模型的推理速度，及不同线程数量（num_threads 参数控制）对推理速度的影响，推理速度用`Real-Time Factor（RTF）`表示，RTF = 推理时间 / 生成语音时长（建议不少于 50 条，条件允许可扩展到 100 条及以上；线程数至少比较 3 组；可用大模型生成 TTS 测试文本）
4. 对比原始模型和 ONNX 优化模型生成语音的质量，使用 Whisper 或其他 ASR 模型评测生成语音的 WER 或 CER，使用 DNSMOS 评测生成语音的客观 MOS 分数（至少对比 50 条语音样本，选取某一线程数量对比即可）
5. 分析 sherpa-onnx 和原始代码在语音前端处理方面实现的差异

**本人选择选题二完成。**

### 问题一：ONNX 模型在推理优化中的核心原理、优势和适用场景

**核心原理：**

ONNX（Open Neural Network Exchange）是一种开放的神经网络模型交换格式。其推理优化的核心原理包括：

1. **计算图优化**：ONNX 将 PyTorch 的动态计算图转换为静态计算图，编译期可进行算子融合（如 Conv+BN+ReLU 合并为单个算子）、常量折叠、冗余节点消除等优化，减少运行时开销。

2. **跨平台部署**：ONNX 定义了统一的算子集和数据格式，一次导出可在多种硬件（CPU/GPU/NPU）和推理引擎（ONNX Runtime、TensorRT、OpenVINO）上运行，无需重写模型代码。

3. **运行时优化**：ONNX Runtime 等推理引擎提供了图级优化（内存规划、并行执行）和算子级优化（针对不同 CPU 指令集的内核选择），充分利用硬件特性。

**优势：**
- **推理速度提升**：静态图 + 算子融合 + 运行时优化，通常比 PyTorch 动态图快 2-5 倍
- **内存占用降低**：优化后的内存规划减少了中间张量的分配
- **多线程支持**：ONNX Runtime 原生支持多线程推理，可通过 `num_threads` 参数灵活控制
- **部署便捷**：无需安装 PyTorch，仅需 onnxruntime 即可运行

**适用场景：**
- 对推理延迟敏感的实时应用（如语音合成、实时翻译）
- 资源受限的边缘设备部署（手机、嵌入式设备）
- 需要跨平台一致性的生产环境

### 问题二：使用 sherpa-onnx 对 MeloTTS 进行 ONNX 模型导出的过程和关键步骤

sherpa-onnx 提供了将 MeloTTS 的 VITS 模型导出为 ONNX 格式的工具。关键步骤如下：

1. **环境准备**：安装 sherpa-onnx 及其依赖（onnx、onnxruntime）
2. **模型导出**：使用 `sherpa-onnx` 提供的导出脚本，将 PyTorch 模型权重转换为 ONNX 格式。导出过程中，VITS 模型的文本编码器、时长预测器、Flow 模块和 HiFi-GAN 解码器分别被序列化为 ONNX 算子
3. **模型优化**：可选地使用 `onnxruntime` 的图优化工具对导出的模型进行进一步优化（算子融合、量化等）
4. **前端适配**：sherpa-onnx 实现了自己的前端处理管线（lexicon + FST），需要准备 `tokens.txt`、`lexicon.txt`、`dict/` 等资源文件

本实验直接使用 sherpa-onnx 官方提供的预训练中文 ONNX 模型（`vits-melo-tts-zh_en`），该模型已包含完整的模型文件和前端资源。

In [1]:
### 问题三：推理速度对比（RTF）
import os, time, json
import numpy as np
import soundfile as sf

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7890'

# 50 条测试文本
test_texts = [
    "今天天气真好，适合出去散步。", "人工智能正在改变我们的生活方式。", "这道菜的味道非常鲜美，我很喜欢。",
    "请把窗户关一下，外面太吵了。", "明天上午九点有一个重要的会议。", "中国的经济发展速度令人瞩目。",
    "这部电影的剧情非常引人入胜。", "学习一门新的语言需要长期的坚持。", "春天来了，公园里的花都开了。",
    "请问去火车站怎么走？", "科技的进步让世界变得更加紧密。", "他每天早上都会跑步锻炼身体。",
    "这本书的内容非常有深度和见解。", "我们需要保护环境，减少污染。", "音乐能够治愈人的心灵。",
    "她在比赛中获得了金牌，为国争光。", "互联网让信息传播变得更加迅速。", "老师耐心地为学生解答问题。",
    "这个项目的进展比预期要快得多。", "健康饮食是保持身体健康的关键。", "机器学习在医疗领域有广泛的应用。",
    "他是一位非常有经验的工程师。", "城市的夜景灯火辉煌，十分壮观。", "阅读可以开阔视野，增长知识。",
    "我们应该珍惜身边的每一个人。", "云计算技术正在改变企业的运营方式。", "这家餐厅的环境非常优雅舒适。",
    "读书使人充实，思考使人深刻。", "旅行是了解不同文化的最好方式。", "坚持锻炼能够增强体质，预防疾病。",
    "她在音乐方面有着极高的天赋。", "创新是推动社会进步的重要动力。", "这部电影值得一看，非常感人。",
    "他为了梦想不懈努力，终于成功了。", "教育是国家发展的基石。", "秋天的树叶变成了金黄色，非常美丽。",
    "我们需要学会管理自己的时间。", "这个城市的历史可以追溯到两千年前。", "合作是实现共赢的最佳途径。",
    "冬天的早晨，空气格外清新。", "他在演讲中表达了自己的观点。", "数字化转型是企业发展的必然趋势。",
    "父母的爱是世界上最伟大的爱。", "运动可以释放压力，让人更加自信。", "她用心制作了一件精美的手工艺品。",
    "人工智能技术在自动驾驶中有重要应用。", "志愿服务是一种美好的社会行为。", "春天是播种希望的季节。",
    "他在科研领域取得了重大突破。", "读书破万卷，下笔如有神。",
]

os.makedirs('results_hw4', exist_ok=True)

# ---- PyTorch 推理 ----
from melo.api import TTS
model = TTS(language='ZH', device='cuda:0', use_hf=False, use_ms=True)
speaker_ids = model.hps.data.spk2id

pytorch_rtf = []
for i, text in enumerate(test_texts):
    out = f'results_hw4/pytorch_{i:03d}.wav'
    t0 = time.time()
    model.tts_to_file(text, speaker_ids['ZH'], out, speed=1.0, quiet=True)
    t1 = time.time()
    audio, sr = sf.read(out)
    pytorch_rtf.append((t1 - t0) / (len(audio) / sr))

print(f"PyTorch: mean RTF = {np.mean(pytorch_rtf):.4f}, median = {np.median(pytorch_rtf):.4f}")

# ---- sherpa-onnx 推理（不同线程数） ----
import sherpa_onnx

model_dir = 'vits-melo-tts-zh_en'
results = {'pytorch': {'mean_rtf': float(np.mean(pytorch_rtf)), 'median_rtf': float(np.median(pytorch_rtf))}}

for num_threads in [1, 2, 4]:
    tts_config = sherpa_onnx.OfflineTtsConfig(
        model=sherpa_onnx.OfflineTtsModelConfig(
            vits=sherpa_onnx.OfflineTtsVitsModelConfig(
                model=f'{model_dir}/model.onnx', tokens=f'{model_dir}/tokens.txt',
                lexicon=f'{model_dir}/lexicon.txt', dict_dir=f'{model_dir}/dict',
            ),
            provider='cpu', num_threads=num_threads,
        ), max_num_sentences=2,
    )
    tts_onnx = sherpa_onnx.OfflineTts(tts_config)
    onnx_rtf = []
    for i, text in enumerate(test_texts):
        out = f'results_hw4/onnx_t{num_threads}_{i:03d}.wav'
        t0 = time.time()
        audio_obj = tts_onnx.generate(text, sid=0, speed=1.0)
        t1 = time.time()
        sf.write(out, audio_obj.samples, audio_obj.sample_rate)
        onnx_rtf.append((t1 - t0) / (len(audio_obj.samples) / audio_obj.sample_rate))
    results[f'onnx_t{num_threads}'] = {'mean_rtf': float(np.mean(onnx_rtf)), 'median_rtf': float(np.median(onnx_rtf))}
    print(f"ONNX (t={num_threads}): mean RTF = {np.mean(onnx_rtf):.4f}, median = {np.median(onnx_rtf):.4f}")

# 汇总
import pandas as pd
df = pd.DataFrame(results).T
df['speedup'] = df['mean_rtf'].iloc[0] / df['mean_rtf']
print("\n===== 推理速度对比 =====")
print(df)

with open('results_hw4/results.json', 'w') as f:
    json.dump(results, f, indent=2)

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.23). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Building prefix dict from the default dictionary ...


Loading model from cache C:\Users\Lenovo\AppData\Local\Temp\jieba.cache


...

Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


PyTorch: mean RTF = 0.1630, median = 0.0610


ONNX (t=1): mean RTF = 0.6351, median = 0.6273


ONNX (t=2): mean RTF = 0.3476, median = 0.3457


ONNX (t=4): mean RTF = 0.2077, median = 0.2061

===== 推理速度对比 =====
         mean_rtf  median_rtf   speedup
pytorch  0.162950    0.060992  1.000000
onnx_t1  0.635114    0.627291  0.256569
onnx_t2  0.347564    0.345662  0.468835
onnx_t4  0.207693    0.206087  0.784571


In [2]:
### 问题四：语音质量对比（DNSMOS + Whisper CER）
import os, json
import numpy as np
import soundfile as sf
import librosa

test_texts = [
    "今天天气真好，适合出去散步。", "人工智能正在改变我们的生活方式。", "这道菜的味道非常鲜美，我很喜欢。",
    "请把窗户关一下，外面太吵了。", "明天上午九点有一个重要的会议。", "中国的经济发展速度令人瞩目。",
    "这部电影的剧情非常引人入胜。", "学习一门新的语言需要长期的坚持。", "春天来了，公园里的花都开了。",
    "请问去火车站怎么走？", "科技的进步让世界变得更加紧密。", "他每天早上都会跑步锻炼身体。",
    "这本书的内容非常有深度和见解。", "我们需要保护环境，减少污染。", "音乐能够治愈人的心灵。",
    "她在比赛中获得了金牌，为国争光。", "互联网让信息传播变得更加迅速。", "老师耐心地为学生解答问题。",
    "这个项目的进展比预期要快得多。", "健康饮食是保持身体健康的关键。", "机器学习在医疗领域有广泛的应用。",
    "他是一位非常有经验的工程师。", "城市的夜景灯火辉煌，十分壮观。", "阅读可以开阔视野，增长知识。",
    "我们应该珍惜身边的每一个人。", "云计算技术正在改变企业的运营方式。", "这家餐厅的环境非常优雅舒适。",
    "读书使人充实，思考使人深刻。", "旅行是了解不同文化的最好方式。", "坚持锻炼能够增强体质，预防疾病。",
    "她在音乐方面有着极高的天赋。", "创新是推动社会进步的重要动力。", "这部电影值得一看，非常感人。",
    "他为了梦想不懈努力，终于成功了。", "教育是国家发展的基石。", "秋天的树叶变成了金黄色，非常美丽。",
    "我们需要学会管理自己的时间。", "这个城市的历史可以追溯到两千年前。", "合作是实现共赢的最佳途径。",
    "冬天的早晨，空气格外清新。", "他在演讲中表达了自己的观点。", "数字化转型是企业发展的必然趋势。",
    "父母的爱是世界上最伟大的爱。", "运动可以释放压力，让人更加自信。", "她用心制作了一件精美的手工艺品。",
    "人工智能技术在自动驾驶中有重要应用。", "志愿服务是一种美好的社会行为。", "春天是播种希望的季节。",
    "他在科研领域取得了重大突破。", "读书破万卷，下笔如有神。",
]

# ---- DNSMOS 评估（需要 16kHz 采样率） ----
from speechmos import dnsmos

def evaluate_dnsmos(audio_files, label):
    scores_list = []
    for fpath in audio_files:
        if os.path.exists(fpath):
            audio, sr = sf.read(fpath)
            if sr != 16000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            scores = dnsmos.run(audio, 16000)
            scores_list.append(scores)
    if scores_list:
        mean_scores = {k: float(np.mean([s[k] for s in scores_list])) for k in scores_list[0].keys()}
        print(f"\n{label} DNSMOS (N={len(scores_list)}):")
        for k, v in mean_scores.items():
            print(f"  {k}: {v:.4f}")
        return mean_scores
    return None

pytorch_files = [f'results_hw4/pytorch_{i:03d}.wav' for i in range(50)]
onnx_files = [f'results_hw4/onnx_t2_{i:03d}.wav' for i in range(50)]

pytorch_dns = evaluate_dnsmos(pytorch_files, "PyTorch")
onnx_dns = evaluate_dnsmos(onnx_files, "ONNX (t=2)")

# ---- Whisper CER 评估 ----
try:
    import whisper, editdistance
    whisper_model = whisper.load_model("base", device="cpu")

    def compute_cer(ref, hyp):
        return editdistance.eval(ref, hyp) / max(len(ref), 1)

    def evaluate_whisper_cer(audio_files, texts, label):
        cers = []
        for fpath, ref_text in zip(audio_files, texts):
            if os.path.exists(fpath):
                result = whisper_model.transcribe(fpath, language="Chinese")
                hyp_text = result["text"].strip()
                cer = compute_cer(ref_text, hyp_text)
                cers.append(cer)
        mean_cer = np.mean(cers)
        print(f"\n{label} Whisper CER (N={len(cers)}): {mean_cer:.4f}")
        return mean_cer

    pytorch_cer = evaluate_whisper_cer(pytorch_files, test_texts, "PyTorch")
    onnx_cer = evaluate_whisper_cer(onnx_files, test_texts, "ONNX (t=2)")
except ImportError as e:
    print(f"\nWhisper/editdistance 未安装: {e}")

# 汇总
print("\n===== 语音质量对比汇总 =====")
if pytorch_dns and onnx_dns:
    import pandas as pd
    df = pd.DataFrame({'PyTorch': pytorch_dns, 'ONNX (t=2)': onnx_dns})
    print(df.T)


PyTorch DNSMOS (N=50):
  ovrl_mos: 3.2878
  sig_mos: 3.5867
  bak_mos: 4.0852
  p808_mos: 3.7375



ONNX (t=2) DNSMOS (N=50):
  ovrl_mos: 3.0989
  sig_mos: 3.5005
  bak_mos: 3.8727
  p808_mos: 3.6222

Whisper/editdistance 未安装: No module named 'whisper'

===== 语音质量对比汇总 =====
            ovrl_mos  sig_mos   bak_mos  p808_mos
PyTorch     3.287821  3.58667  4.085204  3.737496
ONNX (t=2)  3.098891  3.50050  3.872733  3.622205


### 问题五：sherpa-onnx 和原始代码在语音前端处理方面实现的差异分析

| 方面 | 原始 MeloTTS (PyTorch) | sherpa-onnx |
|------|----------------------|-------------|
| **文本正则化** | 使用 Python 实现（`cn2an`、`pypinyin`、`jieba`），支持中英混合文本的数字转写、标点替换 | 使用 C++ 实现的 FST（有限状态转换器），通过 `date.fst`、`number.fst`、`phone.fst` 等预编译 FST 文件处理日期、数字、电话号码等 |
| **字音转换 (G2P)** | 使用 `pypinyin` 库获取拼音，再通过 `opencpop-strict.txt` 映射表将拼音转换为音素序列 | 使用 `lexicon.txt` 词典文件直接查找汉字到音素的映射，对于词典中未收录的词使用 `dict/` 目录下的拼音字典回退 |
| **BERT 特征提取** | 依赖 `transformers` 库加载 `bert-base-multilingual-uncased` 模型，通过前向推理获取每个字的 BERT 嵌入特征，再通过 `word2ph` 展开到音素级 | 不使用 BERT 模型，完全依赖 VITS 模型自身的文本编码器和时长预测器，推理时无需加载大型语言模型，因此内存占用更小、启动更快 |
| **多音字处理** | 依赖 BERT 上下文理解能力 + `pypinyin` 的默认读音选择，无法自定义 | 通过 `lexicon.txt` 和 `dict/` 可以精确定义每个字的读音，但需要手动维护词典 |
| **中英混合** | `chinese_mix.py` 中使用正则表达式分离中英文片段，分别调用中文 G2P 和英文 G2P（`g2p_en` 库），然后合并音素序列 | 词典中同时包含中英文词条的音素映射，通过 `lexicon.txt` 统一处理，但英文发音质量取决于词典覆盖度 |
| **依赖复杂度** | 需要 PyTorch、transformers、jieba、pypinyin、cn2an、g2p_en 等多个 Python 库 | 仅需 `sherpa-onnx` 一个库（底层为 C++），所有前端资源打包为静态文件，部署简单 |
| **推理速度** | BERT 前向推理是主要瓶颈之一，且 Python G2P 处理有额外开销 | 无 BERT 依赖，前端处理为纯 C++ 实现，速度快 |

**总结**：sherpa-onnx 的前端处理采用"词典 + FST"的轻量级方案，去除了对 BERT 等大型语言模型的依赖，显著降低了内存占用和启动时间，但牺牲了部分多音字和上下文理解的准确性。原始 MeloTTS 的前端依赖 BERT 提供丰富的上下文语义特征，在多音字和中英混合场景下理论上更准确，但推理开销更大。